In [ ]:
!pip install -q autogen-ext[openai] autogen-agentchat requests

import asyncio
import requests
from typing import Annotated
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat #for a round robin group chat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelCapabilities, ModelInfo
from autogen_core.tools import FunctionTool



#Refer to the API keys
GROQ_API_KEY="my-secret"
WEATHER_API_KEY="my-weather-secret"


#Weather Function

def get_weather(city:Annotated[str,"city name"])->str:
  """Get the weather for a city"""
  try:
      city=city.lower().title()
      url=f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}"
      params={
          "q":city,
          "appid":WEATHER_API_KEY,
          "units":"metric"
      }
      response=requests.get(url,params=params,timeout=10)
      data=response.json()

      #API Validation Logic
      if str(data.get("cod"))!="200":
        return f"Invalid city name {city}"

      temp=data["main"]["temp"]
      feels_like=data["main"]["feels_like"]
      wind_speed=data["wind"]["speed"]
      condition=data["weather"][0]["description"]
      pressure=data["main"]["pressure"]
      humidity=data["main"]["humidity"]
      description=data["weather"][0]["description"]

      return (
            f"Weather Profile for {city}:\n"
            f"- Temperature: {temp}°C (Feels like: {feels_like}°C)\n"
            f"- Conditions: {condition} ({description})\n"
            f"- Wind Speed: {wind_speed} m/s\n"
            f"- Humidity: {humidity}% | Pressure: {pressure} hPa"
      )

  except requests.exceptions.RequestException as e:
          return f"Error:{e}"


##Model setup

#Model Capabilities are optional
groq_capabilities=ModelCapabilities(
                      vision=False,
                      function_calling=True,
                      json_output=True

                      )

# Define ModelInfo for the custom Groq model by passing capabilities directly as arguments
groq_model_info = ModelInfo(
    name="llama-3.3-70b-versatile",
    family="llama",
    vision=groq_capabilities.get("vision", False),
    function_calling=groq_capabilities.get("function_calling", True),
    json_output=groq_capabilities.get("json_output", True),
    structured_output=False
)

model_client = OpenAIChatCompletionClient(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model_info=groq_model_info,
    extra_create_args={"parallel_tool_calls": False}
)

##Call weather Agent
weather_agent=AssistantAgent(
    name="Weather_Agent",
    model_client=model_client,
    tools=[FunctionTool(get_weather,description="Get weather for a city")],
    system_message=("You are a weather assitant.\n"
                    "When user ask about the weather of a particular city, always use the get_weather tool.\n"
                    "then explain the results clearly and precisely"
                 )
              )

##Run the code

async def run():
  print("======Weather Assistant=====\n")

  while True:
    city=input("Enter City name(or type 'exit'):")

    if city.lower()=="exit":
      print('GoodBye')
      break
    task=f"What is the weather{city}?"

    termination=MaxMessageTermination(max_messages=4)
    team=RoundRobinGroupChat(
                             [weather_agent],
                             termination_condition=termination

                           )

    result=await team.run(task=task)

    for message in result.messages:
      if message.source=="Weather_Agent":
         print(f"Assistant:{message.content}")


await run()

======Weather Assistant=====

Enter City name(or type 'exit'):Toronto
Assistant:[FunctionCall(id='cxp6bmcqj', arguments='{"city":"Toronto"}', name='get_weather')]
Assistant:[FunctionExecutionResult(content='Weather Profile for Toronto:\n- Temperature: 18.13°C (Feels like: 18.27°C)\n- Conditions: overcast clouds (overcast clouds)\n- Wind Speed: 0.89 m/s\n- Humidity: 87% | Pressure: 1006 hPa', name='get_weather', call_id='cxp6bmcqj', is_error=False)]
Assistant:Weather Profile for Toronto:
- Temperature: 18.13°C (Feels like: 18.27°C)
- Conditions: overcast clouds (overcast clouds)
- Wind Speed: 0.89 m/s
- Humidity: 87% | Pressure: 1006 hPa
Assistant:The current weather in Toronto is overcast clouds with a temperature of 18.13°C, feeling like 18.27°C. The wind speed is 0.89 m/s, and the humidity is 87% with a pressure of 1006 hPa.
Assistant:
Enter City name(or type 'exit'):London
Assistant:[FunctionCall(id='mfask34q0', arguments='{"city":"London"}', name='get_weather')]
Assistant:[Function

AutoGen Implementation
